# Remote Cleanup 02: `device.cleanup()` and Constructor Auto-Cleanup

Focused manual test for global cleanup and constructor auto-cleanup. Cells are deliberately direct: re-run START 2 to create resources, then run one trigger and one error cell at a time.

In [ ]:
import gc
import json
import os
from pathlib import Path

import numpy as np

REMOTE_IP = os.environ.get("PYNQ_REMOTE_DEVICES", "192.168.2.197").split(",")[0].strip()
os.environ["PYNQ_REMOTE_DEVICES"] = REMOTE_IP
OVERLAY_PATH = "/workspace/phd/PYNQ.remote-dev/applications/PYNQ/tests/resizer.xsa"

import pynq

print("REMOTE_IP:", REMOTE_IP)
print("OVERLAY_PATH:", OVERLAY_PATH)

In [ ]:
# START 1: create a RemoteDevice through normal PYNQ import/probe.
# Re-run this cell when you want constructor auto_cleanup=True to run again.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")

remote_devices = [d for d in pynq.Device.devices if d.has_capability("REMOTE")]
if not remote_devices:
    raise RuntimeError("No RemoteDevice found. Check PYNQ_REMOTE_DEVICES and board connectivity.")

device = remote_devices[0]
print("device:", device)

In [ ]:
# START 2: load the overlay and create one MMIO, one buffer, and optional GPIO.
# Re-run START 1 first if you want a fresh device/constructor cleanup.
overlay = pynq.Overlay(OVERLAY_PATH, device=device)
resizer = overlay.resize_accel_0
resizer.read(0)

mmio = resizer.mmio
mmio_id = mmio._remote_map.mmio_id

buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
buffer[:] = np.arange(16, dtype=np.uint32)
buffer.flush()
buffer_id = buffer.buffer_id

gpio = None
gpio_id = None
gpio_pin = None
gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=device)
npins = pynq.GPIO.get_gpio_npins(device=device)
if base_path and npins:
    gpio_pin = pynq.GPIO.get_gpio_pin(0, device=device)
    gpio_path = f"/sys/class/gpio/gpio{gpio_pin}"
    if device.exists_file(gpio_path).exists:
        print("GPIO already exported, skipping GPIO object:", gpio_path)
    else:
        gpio = pynq.GPIO(gpio_pin, "in", device=device)
        gpio.read()
        gpio_id = gpio._gpio_id
else:
    print("GPIO sysfs base not available; GPIO cells will skip.")

print("MMIO id:  ", mmio_id)
print("Buffer id:", buffer_id)
print("GPIO id:  ", gpio_id, "path:", gpio_path)

## Explicit `device.cleanup()`

Run START 2, then run the cleanup trigger and each stale error cell. You should see `Object stale`.

In [ ]:
# Cleanup trigger: run START 2 first, then this cell.
print("old MMIO id:  ", mmio_id)
print("old Buffer id:", buffer_id)
print("old GPIO id:  ", gpio_id)
response = device.cleanup()
print(response)
print("You should see Cleanup Request Received in the target logs.")

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ MMIO object should fail.
mmio.read(0)

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ Buffer object should fail.
buffer.physical_address

In [ ]:
# Error test: after global cleanup, using the old normal PYNQ GPIO object should fail.
if gpio_id is None:
    print("Skipping GPIO stale test because no GPIO object was created.")
else:
    gpio.read()

In [ ]:
# Release-style normal PYNQ cleanup calls should be safe even after global cleanup.
# Run this after the cleanup trigger. It should not error.
mmio.close()
buffer.freebuffer()
if gpio is not None:
    gpio.release()
print("Release-style PYNQ cleanup calls completed.")

In [ ]:
# Fresh allocation after cleanup should not alias the old handles.
fresh_overlay = pynq.Overlay(OVERLAY_PATH, device=device)
fresh_resizer = fresh_overlay.resize_accel_0
fresh_resizer.read(0)
fresh_mmio = fresh_resizer.mmio
fresh_mmio_id = fresh_mmio._remote_map.mmio_id

fresh_buffer = device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
fresh_buffer_id = fresh_buffer.buffer_id

print("old -> fresh MMIO:  ", mmio_id, "->", fresh_mmio_id)
print("old -> fresh Buffer:", buffer_id, "->", fresh_buffer_id)
assert fresh_mmio_id != mmio_id
assert fresh_buffer_id != buffer_id
print("Fresh handles did not alias old handles.")

## Constructor Auto-Cleanup Across Kernel Restart

Run the seed cell, restart the kernel, then run Setup and the verification cell.

In [ ]:
# Cross-kernel setup: create orphan-prone resources without constructor auto-cleanup.
# Run Setup first. Run this cell, check the target logs, then restart the kernel.
from pynq.pl_server.remote_device import RemoteDevice
seed_device = RemoteDevice(ip_addr=REMOTE_IP, auto_cleanup=False)
seed_overlay = pynq.Overlay(OVERLAY_PATH, device=seed_device)
seed_resizer = seed_overlay.resize_accel_0
seed_resizer.read(0)
seed_mmio = seed_resizer.mmio
seed_buffer = seed_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
seed_buffer[:] = np.arange(16, dtype=np.uint32)
seed_buffer.flush()

seed_gpio = None
seed_gpio_id = None
seed_gpio_path = None
base_path = pynq.GPIO.get_gpio_base_path(device=seed_device)
npins = pynq.GPIO.get_gpio_npins(device=seed_device)
if base_path and npins:
    seed_gpio_pin = pynq.GPIO.get_gpio_pin(0, device=seed_device)
    seed_gpio_path = f"/sys/class/gpio/gpio{seed_gpio_pin}"
    if not seed_device.exists_file(seed_gpio_path).exists:
        seed_gpio = pynq.GPIO(seed_gpio_pin, "in", device=seed_device)
        seed_gpio.read()
        seed_gpio_id = seed_gpio._gpio_id

STATE_FILE = Path("/tmp/pynq_remote_cleanup_constructor_state.json")
state = {
    "mmio_id": seed_mmio._remote_map.mmio_id,
    "buffer_id": seed_buffer.buffer_id,
    "gpio_id": seed_gpio_id,
    "gpio_path": seed_gpio_path,
}
STATE_FILE.write_text(json.dumps(state, indent=2))
print(state)
print("Restart the kernel now. After restart, run Setup and then the constructor verification cell.")

In [ ]:
# Constructor auto-cleanup verification using normal PYNQ discovery.
# Run Setup first after the restart, then this cell.
STATE_FILE = Path("/tmp/pynq_remote_cleanup_constructor_state.json")
state = json.loads(STATE_FILE.read_text())
print("old handles saved before restart:", state)

# Normal PYNQ probing constructs a RemoteDevice with auto_cleanup=True.
if hasattr(pynq.Device, "_active_device"):
    delattr(pynq.Device, "_active_device")
if hasattr(pynq.Device, "_devices"):
    delattr(pynq.Device, "_devices")
cleanup_device = [d for d in pynq.Device.devices if d.has_capability("REMOTE")][0]
print("constructed auto-cleanup device:", cleanup_device)
print("Check target logs: constructor cleanup should have run before any fresh resources below.")

fresh_overlay = pynq.Overlay(OVERLAY_PATH, device=cleanup_device)
fresh_resizer = fresh_overlay.resize_accel_0
print("fresh register read:", fresh_resizer.read(0))
fresh_buffer = cleanup_device.allocate(shape=(16,), dtype=np.uint32, cacheable=1)
fresh_buffer[:] = np.arange(16, dtype=np.uint32)
fresh_buffer.flush()
print("fresh buffer physical address:", fresh_buffer.physical_address)
fresh_buffer.freebuffer()